# CRU TS vs ERA5: India temperature comparison (2000-2018)

How well do [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels)
and [CRU TS](https://crudata.uea.ac.uk/cru/data/hrg/) agree on monthly temperature
over India? They are two independent records:

- **ERA5** -- ECMWF reanalysis at 0.25\u00b0, pulled with `era5ify_bbox`
- **CRU TS v4.07** -- station-interpolated observations from UEA at 0.5\u00b0, pulled with `cru_ts_bbox`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import varunayan as v
from varunayan.cru_ts import list_cru_ts_variables, cru_ts_bbox

## CRU TS variable dictionary

Each CRU TS variable is listed with its ERA5 and HadEX3 equivalents.

In [ ]:
list_cru_ts_variables()[["variable", "name", "unit", "era5_equivalent"]]

## Download

ERA5 requires a CDS account. CRU TS downloads directly from UEA (no credentials).

In [ ]:
india_bbox = dict(north=37, south=6, east=98, west=68)

era5_tmp = v.era5ify_bbox(
    request_id="india_era5_tmp_monthly",
    variables="2m_temperature",
    start_date="2000-01-01", end_date="2018-12-31",
    frequency="monthly", resolution=0.5,
    **india_bbox,
)

cru_tmp = cru_ts_bbox(
    variable="tmp",
    start_year=2000, end_year=2018,
    **india_bbox,
)

print(f"ERA5: {len(era5_tmp)} rows")
print(f"CRU TS: {len(cru_tmp)} rows")

## Monthly time series

Spatial mean temperature across all India grid cells, month by month.

In [ ]:
era5_monthly = (
    era5_tmp.groupby(["year", "month"])["value"].mean()
    .reset_index()
    .assign(source="ERA5",
            date=lambda d: pd.to_datetime(d["year"].astype(str) + "-" + d["month"].astype(str) + "-01"))
)

cru_monthly = (
    cru_tmp.groupby(["year", "month"])["value"].mean()
    .reset_index()
    .assign(source="CRU TS",
            date=lambda d: pd.to_datetime(d["year"].astype(str) + "-" + d["month"].astype(str) + "-01"))
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(era5_monthly["date"], era5_monthly["value"], color="#E74C3C", alpha=0.4, lw=0.5)
ax.plot(cru_monthly["date"], cru_monthly["value"], color="#2E86AB", alpha=0.4, lw=0.5)

from statsmodels.nonparametric.smoothers_lowess import lowess
for df, color, label in [(era5_monthly, "#E74C3C", "ERA5"), (cru_monthly, "#2E86AB", "CRU TS")]:
    sm = lowess(df["value"], np.arange(len(df)), frac=0.07)
    ax.plot(df["date"], sm[:, 1], color=color, lw=1.5, label=label)

ax.legend(loc="upper left")
ax.set_ylabel("Mean temperature (\u00b0C)")
ax.set_title("India: Monthly Mean Temperature \u2014 ERA5 vs CRU TS (2000-2018)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Scatter comparison

In [ ]:
temp_wide = era5_monthly[["year", "month", "value"]].rename(columns={"value": "ERA5"}).merge(
    cru_monthly[["year", "month", "value"]].rename(columns={"value": "CRU_TS"}),
    on=["year", "month"],
)

cor_val = temp_wide["ERA5"].corr(temp_wide["CRU_TS"])
bias = (temp_wide["ERA5"] - temp_wide["CRU_TS"]).mean()

fig, ax = plt.subplots(figsize=(5, 5))
lims = [temp_wide[["ERA5", "CRU_TS"]].min().min() - 1,
        temp_wide[["ERA5", "CRU_TS"]].max().max() + 1]
ax.plot(lims, lims, "--", color="grey", lw=0.8)
ax.scatter(temp_wide["CRU_TS"], temp_wide["ERA5"], alpha=0.5, s=12, color="#34495E")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.set_xlabel("CRU TS (\u00b0C)")
ax.set_ylabel("ERA5 (\u00b0C)")
ax.set_title(f"ERA5 vs CRU TS: Spatial Mean Monthly Temperature\n"
             f"r = {cor_val:.3f}   bias (ERA5 \u2212 CRU TS) = {bias:+.2f}\u00b0C")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Seasonal cycle

Averaged over 2000-2018, the seasonal cycles nearly overlap.

In [ ]:
import calendar

temp_all = pd.concat([era5_monthly, cru_monthly], ignore_index=True)
seasonal = temp_all.groupby(["source", "month"])["value"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
for source, color in [("ERA5", "#E74C3C"), ("CRU TS", "#2E86AB")]:
    sub = seasonal[seasonal["source"] == source]
    ax.plot(sub["month"], sub["value"], color=color, lw=1.5, marker="o", ms=5, label=source)

ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_ylabel("Mean temperature (\u00b0C)")
ax.set_title("India: Seasonal Temperature Cycle (2000-2018 mean)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Spatial bias map

We snap ERA5 (0.25\u00b0) onto the CRU TS 0.5\u00b0 grid and compare cell by cell.

In [ ]:
era5_spatial = era5_tmp.copy()
era5_spatial["lat_bin"] = np.floor(era5_spatial["latitude"] / 0.5) * 0.5 + 0.25
era5_spatial["lon_bin"] = np.floor(era5_spatial["longitude"] / 0.5) * 0.5 + 0.25
era5_spatial = era5_spatial.groupby(["lat_bin", "lon_bin"])["value"].mean().reset_index(name="era5_mean")

cru_spatial = cru_tmp.groupby(["latitude", "longitude"])["value"].mean().reset_index(name="cru_mean")

spatial = era5_spatial.merge(
    cru_spatial, left_on=["lat_bin", "lon_bin"], right_on=["latitude", "longitude"],
)
spatial["bias"] = spatial["era5_mean"] - spatial["cru_mean"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
kw = dict(s=15, edgecolors="none")

sc1 = axes[0].scatter(spatial["lon_bin"], spatial["lat_bin"], c=spatial["era5_mean"],
                       cmap="RdYlBu_r", vmin=10, vmax=36, **kw)
axes[0].set_title("ERA5 Annual Mean")

sc2 = axes[1].scatter(spatial["lon_bin"], spatial["lat_bin"], c=spatial["cru_mean"],
                       cmap="RdYlBu_r", vmin=10, vmax=36, **kw)
axes[1].set_title("CRU TS Annual Mean")

blim = max(abs(spatial["bias"].min()), abs(spatial["bias"].max()))
sc3 = axes[2].scatter(spatial["lon_bin"], spatial["lat_bin"], c=spatial["bias"],
                       cmap="RdBu_r", vmin=-blim, vmax=blim, **kw)
axes[2].set_title("ERA5 \u2212 CRU TS")

for ax in axes:
    ax.set_aspect("equal")

fig.colorbar(sc1, ax=axes[:2], label="\u00b0C", shrink=0.7)
fig.colorbar(sc3, ax=axes[2], label="\u00b0C", shrink=0.7)
plt.tight_layout()
plt.show()

## Annual trend

In [ ]:
annual = temp_all.groupby(["source", "year"])["value"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
for source, color in [("ERA5", "#E74C3C"), ("CRU TS", "#2E86AB")]:
    sub = annual[annual["source"] == source]
    ax.plot(sub["year"], sub["value"], color=color, lw=1, marker="o", ms=3, label=source)
    z = np.polyfit(sub["year"], sub["value"], 1)
    ax.plot(sub["year"], np.polyval(z, sub["year"]), color=color, lw=1, ls="--")

ax.legend()
ax.set_ylabel("Annual mean temperature (\u00b0C)")
ax.set_title("India: Annual Temperature Trend \u2014 ERA5 vs CRU TS\nDashed = linear trend")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## When to use which

| Feature | CRU TS v4.07 | ERA5 |
|---------|-------------|------|
| Resolution | 0.5\u00b0 | 0.25\u00b0 (or coarser) |
| Coverage | 1901-2022 | 1940-present |
| Frequency | Monthly | Hourly, daily, monthly |
| Source | Station interpolation (land) | Reanalysis (model + obs, land + ocean) |
| Credentials | No (HTTP download) | Yes (CDS account) |
| Variables | tmp, tmx, tmn, pre, vap, cld, wet, frs, dtr, pet | 100+ incl. wind, radiation, soil moisture |

CRU TS is the simpler choice for long historical baselines without CDS
credentials, or when monthly land-only temperature is enough. ERA5 wins for
sub-monthly frequency, ocean coverage, or variables beyond temperature.